# 10. 综合任务

AGDR VLA 入门教程 · 第 2 章 · 第 2 周

- **学习目标**：梳理危险操作和 Agent 协作原则，综合运用本章内容完成作业
- **前置知识**：第 01–09 节
- **预计学习时长**：3 小时（前半部分 30 分钟，作业约 2.5 小时）

本节先讲两个收尾知识点，再完成全章唯一的一组作业：

1. **作业一**：写一个日志分析脚本，用 Pull Request 提交
2. **作业二**：完成 Agent 选择题，覆盖第 01–10 节
3. **作业三**：提交 Issue

## 危险操作清单

| 操作 | 风险 | 执行前先做什么 |
| --- | --- | --- |
| `rm -rf <路径>` | 直接删除，不进回收站 | 先用 `ls <同样的路径>` 看清楚 |
| 路径里带变量：`rm -rf "$DIR/"*` | 变量为空时，会变成删除根目录下的内容 | 脚本里 `set -u`，并先 `echo` 出来 |
| `命令 > 文件` | 立刻清空原文件 | 确认文件名，不确定就用 `>>` |
| `cp` / `mv` 同名文件 | 默默覆盖 | 加 `-i` |
| `chmod -R 777`、`chown -R` | 大范围改权限，难以恢复 | 只改需要的文件和权限 |
| `sudo <任何命令>` | 没有权限检查兜底 | 先弄清为什么需要 `sudo` |
| `curl ... \| bash` | 运行没看过的代码 | 确认来源可信，先读内容 |
| `kill -9` | 程序来不及保存 | 先用普通 `kill`；确认 PID 是自己的进程 |
| `rsync --delete` | 删除目标中多余的文件 | 先加 `-n` 试运行 |
| `dd of=/dev/...`、`mkfs` | 直接抹掉磁盘 | 反复确认设备名；不懂就不要运行 |

**三个习惯能避免大多数事故：**

1. **先看再动**：用 `pwd`、`ls`、`echo` 确认当前位置和目标
2. **先备份**：改配置前 `cp 文件 文件.bak`；虚拟机拍快照
3. **看清提示符**：我是谁？（`$` 还是 `#`）在哪台机器上？在哪个目录？

## 与 Agent 协作使用终端

Agent 可以写命令，甚至直接运行命令，这让效率更高，也让风险更大。

**Agent 适合帮你：**

- 解释命令和报错的含义
- 把需求写成管道或脚本的**草稿**
- 总结很长的日志或命令输出

**使用原则：**

1. **先理解，再执行**：每个选项都要能说清含义，不懂就接着问
2. **危险命令先确认**：遇到上面清单里的命令，先问清后果和影响范围
3. **逐段验证**：长管道一段一段地运行
4. **运行后检查**：用 `ls`、`cat`、`echo $?` 自己确认结果
5. **敏感信息不外传**：不要把私钥、令牌、密码、`.env` 内容发给 Agent，粘贴日志前也要先检查
6. **不要盲信**：Agent 也可能编造不存在的选项，用 `--help` 或 `man` 核实

### 提示词示例

```text
请逐段解释这条命令，每个选项是什么意思，有什么风险，不要运行它：
<粘贴命令>
```

```text
我运行命令后得到下面的报错。请先解释报错的含义和可能的原因，
告诉我该运行什么命令来确认，不要直接给修复方案：
<粘贴报错>
```

```text
我的脚本目的是 xxx，请检查有没有未加引号的变量或其他隐患，只指出问题，不要改写：
<粘贴脚本>
```

> **做本节作业时**，仓库的 Agent 只讲思路和提示，不会替你写。这是有意的安排。

## 作业一：日志分析脚本

写一个 Shell 脚本，分析一份模拟的训练日志，并通过 Pull Request 提交。

### 日志格式

本章目录的 `data/` 里有两份日志：

- `data/train_small.log`：16 行，用来调试，下面给出了它的正确结果
- `data/train_full.log`：完整日志，结果要写进 Pull Request

先看一下日志的内容：

In [ ]:
%%bash
head -n 7 data/train_small.log

每行依次是日期、时间、级别、内容。`ERROR` 行的第 4 列是错误类型。

### 要求

脚本接受**一个参数**：日志文件的路径。运行后输出下面 7 行，**格式必须完全一致**：

| 字段 | 含义 |
| --- | --- |
| `lines` | 文件总行数 |
| `info` / `warn` / `error` | 对应级别的行数 |
| `best_loss` | 所有 `loss=` 中的最小值 |
| `best_step` | 最小 loss 所在行的 `step` |
| `top_error` | 出现次数最多的错误类型 |

`train_small.log` 的正确输出：

```text
lines: 16
info: 11
warn: 2
error: 3
best_loss: 1.1093
best_step: 400
top_error: dataloader_timeout
```

另外，脚本还要满足：

| 情况 | 行为 |
| --- | --- |
| 参数个数不是 1 | 向 **stderr** 输出用法，退出码为 **1** |
| 文件不存在 | 向 **stderr** 输出错误，退出码为 **2** |
| 正常运行 | 向 stdout 输出结果，退出码为 **0** |

- 第一行是 shebang，文件有可执行权限，路径带空格时也能正常运行
- 只使用 Shell 和本章学过的命令，不使用 Python，也不把结果写死在脚本里

### 提示

- 所需的命令都在第 03、04、06 节，先写好单独的管道，再放进脚本
- 用 `$(...)` 把管道的结果存进变量
- 目录名带空格：在终端里 `cd` 时要加引号

### 提交步骤

1. 同步你的 Fork，并新建分支：`homework/log-report-你的GitHub用户名`
2. 把脚本放到 `2. Linux终端操作/submission/log_report_你的GitHub用户名.sh`
3. 确认可执行权限已经进入 Git：

   ```bash
   chmod +x "2. Linux终端操作/submission/log_report_你的GitHub用户名.sh"
   git add "2. Linux终端操作/submission/log_report_你的GitHub用户名.sh"
   git ls-files -s "2. Linux终端操作/submission/log_report_你的GitHub用户名.sh"
   ```

   输出开头应当是 `100755`。如果是 `100644`，说明权限没有记录下来。
4. 提交、推送，并向原仓库发起 Pull Request，标题格式：`[第2章-作业一] 你的名字`
5. 在 Pull Request 的说明中粘贴以下三项运行结果：
   - 脚本处理 `train_full.log` 的输出
   - 不带参数运行时的输出和 `echo $?` 的结果
   - 传入不存在的文件时的输出和 `echo $?` 的结果

**检查标准：**

- 输出格式和数值都正确，文件名、分支名和 Pull Request 标题符合规则
- 两种错误情况的退出码正确，错误信息输出到 stderr
- 可执行权限为 `100755`，变量都加了双引号，只提交了自己的脚本

## 作业二：Agent 选择题

本章选择题统一在这里做，范围是第 01–10 节的全部内容。

1. 在仓库**根目录**启动 Agent（Codex、Claude Code、DeepSeek Harness 任选其一），直接开始对话
2. 选择「答题模式」和「第 2 章」，按提示作答，直到 Agent 宣布完成
3. 完成后，Agent 会给出一份总结，把它复制到作业三的 Issue 里

要求：

- 自己作答，不要让 Agent 直接给出答案
- 答错的题目要弄懂原因，可以切换到问答模式提问，再回来继续
- 如果 Agent 出的题有明显错误，记录下来，写进作业三的 Issue

## 作业三：提交 Issue

在原仓库新建一个 Issue：

**标题**：`[第2章-作业三] 你的名字`

**正文**：

1. **系统信息**：你用的是虚拟机还是双系统，以及 `uname -sm` 和 `grep PRETTY_NAME /etc/os-release` 的输出
2. **tmux 实操记录**：完成第 09 节的完整练习，并粘贴以下内容：
   - 第 6 步重新登录后，`tmux ls` 的输出
   - 第 8 步回到会话后，屏幕上最后几行 `step` 的内容
3. **作业二的最终结论**与最后一组题目（Agent 给出的总结）
4. **一个问题或建议**：本章中你遇到的问题、不清楚的地方，或者对教程的改进建议
5. 如果发现 Agent 出题或讲解有错误，写出具体题目和你的判断

> 粘贴前检查一下，内容里不要包含密码、令牌或私钥。

## 完成检查表

**知识**

- [ ] 第 1 周：01 终端与文件系统、02 文件操作、03 查看与搜索、04 管道与重定向、05 用户与权限
- [ ] 第 2 周：06 环境变量与 Shell 脚本、07 进程与系统资源、08 软件安装与压缩包、09 远程连接

**作业**

- [ ] 作业一：Pull Request 已提交，`train_small.log` 的输出和给定结果一致
- [ ] 作业二：Agent 判定完成，结论已记录
- [ ] 作业三：Issue 已提交

全部完成后，就可以进入第 3 章了。你之后的所有内容，都会建立在这一章的终端操作之上。